# SatQuery — serve the specialist models from Kaggle

Downloads every pretrained checkpoint, starts `ml/serve.py` across both GPUs, and
opens a public tunnel so the SatQuery backend can call it.

**Nothing is trained here.** Every model is someone else's published work; the
attribution is in `ml/satquery_ml/registry.py` and is returned in every API
response and by `/health`.

| Task | Model | Trained by |
| --- | --- | --- |
| land cover (S1+S2) | BigEarthNet-19 ResNet-50 | BIFOLD / TU Berlin |
| VQA, captioning, phrasing | RSCoVLM-7B (Qwen2.5-VL) | VisionXLab |
| text-to-box grounding | Grounding DINO base | IDEA-Research |
| referring segmentation | SAM 2.1 Hiera-L | Meta AI |
| bi-temporal change | ChangeFormerV6 (LEVIR-CD) | Bandara & Patel |
| optical–SAR fusion | CROMA-base | Fuller et al. |

## Before running

1. **Settings → Accelerator → GPU T4 x2.** The VLM takes `cuda:0`, the five
   specialists share `cuda:1`. One GPU also works (eviction turns on automatically)
   but task switches get slower.
2. **Settings → Internet → On.** Required for the weight downloads and the tunnel.

Total download is roughly 12 GB, most of it the VLM.

In [ ]:
# 1. Confirm the accelerator before downloading 12 GB of weights.
import subprocess

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(index)
    total = torch.cuda.get_device_properties(index).total_memory / 1e9
    print(f"  cuda:{index}  {name}  {total:.1f} GB")

if torch.cuda.device_count() < 2:
    print(
        "\nNOTE: fewer than 2 GPUs. The server will share one card and evict models\n"
        "between tasks, which still works but is slower. Set Accelerator to T4 x2."
    )
print("\nbf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [ ]:
# 2. Get the repo. Set REPO_URL to your remote (or attach the repo as a dataset).
import os
import sys
from pathlib import Path

REPO_URL = os.environ.get("SATQUERY_REPO", "")  # e.g. https://github.com/<you>/satquery.git
REPO_DIR = Path("/kaggle/working/satquery")

if not REPO_DIR.exists():
    if REPO_URL:
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        raise SystemExit(
            "Set REPO_URL above, or attach the repo as a Kaggle dataset and point\n"
            "REPO_DIR at it."
        )
else:
    !cd {REPO_DIR} && git pull --ff-only

ML_DIR = REPO_DIR / "ml"
sys.path.insert(0, str(ML_DIR))
print("ml/ contents:")
for item in sorted(ML_DIR.iterdir()):
    print(" ", item.name)

In [ ]:
# 3. Dependencies. Kaggle ships torch/transformers; these are the extras.
!pip install -q torchgeo timm safetensors "fastapi>=0.115" "uvicorn[standard]>=0.30" 2>&1 | tail -3

# configilm is the official BigEarthNet loader. If it fails to install, the
# adapter falls back to loading the same weights through timm.
!pip install -q configilm 2>&1 | tail -2

# SAM 2 is optional: without it, grounding returns boxes but no masks.
!pip install -q "git+https://github.com/facebookresearch/sam2.git" 2>&1 | tail -2

print("\ninstall step finished (failures above are non-fatal, see notes)")

In [ ]:
# 4. ChangeFormer: the model definition is not on PyPI, so vendor the repo and
#    fetch the LEVIR-CD release checkpoint.
VENDOR = ML_DIR / "vendor" / "ChangeFormer"
CKPT_DIR = VENDOR / "checkpoints" / "ChangeFormer_LEVIR"

if not VENDOR.exists():
    !git clone --depth 1 https://github.com/wgcban/ChangeFormer.git {VENDOR}

CKPT_DIR.mkdir(parents=True, exist_ok=True)
if not (CKPT_DIR / "best_ckpt.pt").exists():
    RELEASE = (
        "https://github.com/wgcban/ChangeFormer/releases/download/v0.1.0/"
        "CD_ChangeFormerV6_LEVIR_b16_lr0.0001_adamw_train_test_200_linear_ce_"
        "multi_train_True_multi_infer_False_shuffle_AB_False_embed_dim_256.zip"
    )
    !wget -q -O /tmp/changeformer.zip "{RELEASE}" && unzip -o -q /tmp/changeformer.zip -d /tmp/cf
    # The archive nests the checkpoint one directory deep; find it wherever it lands.
    !find /tmp/cf -name 'best_ckpt.pt' -exec cp {{}} {CKPT_DIR}/best_ckpt.pt \;

found = (CKPT_DIR / "best_ckpt.pt").exists()
print(f"ChangeFormer checkpoint present: {found}")
if not found:
    print("  change detection will fall back to image differencing, labelled as such")

In [ ]:
# 5. Download and smoke-test each model individually, so a failure names itself
#    instead of surfacing as a dead endpoint later.
import numpy as np

from satquery_ml.bands import BEN_ALL_BANDS, BandStack
from satquery_ml.loader import ModelLoader

loader = ModelLoader()
print("device plan:", loader.devices, "\n")

status = loader.warmup(("landcover", "grounding_detector", "change", "fusion", "vlm"))
for key, state in status.items():
    mark = "ok  " if state == "loaded" else "FAIL"
    print(f"{mark} {key}: {state}")

In [ ]:
# 6. Run one real prediction through each loaded model.
rng = np.random.default_rng(0)
stack = BandStack(
    array=rng.random((120, 120, len(BEN_ALL_BANDS))).astype(np.float32),
    names=BEN_ALL_BANDS,
)
rgb = stack.rgb()

if status.get("landcover") == "loaded":
    evidence = loader.load("landcover").evidence(stack, "Is there water in this scene?")
    print("land cover  ->", evidence.one_word, "|", evidence.findings)

if status.get("grounding_detector") == "loaded":
    evidence = loader.load("grounding_detector").evidence(rgb, "Highlight the water body")
    print("grounding   ->", len(evidence.boxes), "box(es),", len(evidence.masks), "mask(s)")

if status.get("change") == "loaded":
    after = rgb.copy()
    after[30:80, 30:80] = 255
    from satquery_ml.adapters import change as change_module

    probability = loader.load("change").change_probability(rgb, after)
    print("change      ->", change_module.summarise_mask(probability))

if status.get("fusion") == "loaded":
    print("fusion      ->", loader.load("fusion").agreement(stack))

if status.get("vlm") == "loaded":
    print("vlm phrase  ->", loader.load("vlm").phrase(evidence)[:160])

In [ ]:
# 7. Free the notebook's copies before the server loads its own, or the two
#    sets of weights compete for the same VRAM.
import gc

for key in list(loader._adapters):
    loader._adapters[key].unload()
del loader
gc.collect()
torch.cuda.empty_cache()

for index in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(index)
    print(f"cuda:{index}  {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")

In [ ]:
# 8. Start the server as a subprocess and wait for /health.
import json
import time
import urllib.request

PORT = 8100
LOG = Path("/kaggle/working/serve.log")

server = subprocess.Popen(
    [sys.executable, "serve.py", "--port", str(PORT), "--warmup"],
    cwd=str(ML_DIR),
    stdout=LOG.open("w"),
    stderr=subprocess.STDOUT,
)
print(f"serve.py started as pid {server.pid}; warmup loads every model, so allow a few minutes")

health = None
for attempt in range(120):
    if server.poll() is not None:
        print("server exited early. Last 40 log lines:")
        print("\n".join(LOG.read_text().splitlines()[-40:]))
        raise SystemExit(1)
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as reply:
            health = json.loads(reply.read())
        break
    except Exception:
        time.sleep(5)

if health is None:
    print("\n".join(LOG.read_text().splitlines()[-40:]))
    raise SystemExit("server did not become healthy")

print("\nhealthy. Model status:")
for row in health["models"]:
    mark = "loaded  " if row["loaded"] else "not loaded"
    print(f"  {mark} {row['key']:22s} {row['device']:8s} {row['model']}")
    if row.get("loadError"):
        print(f"      error: {row['loadError']}")

In [ ]:
# 9. Exercise every endpoint locally before exposing it publicly.
import base64
import io

from PIL import Image


def post(path, payload):
    request = urllib.request.Request(
        f"http://127.0.0.1:{PORT}{path}",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as reply:
            return json.loads(reply.read())
    except urllib.error.HTTPError as exc:
        return {"error": exc.code, "detail": exc.read().decode()[:300]}


def png_b64(array):
    buffer = io.BytesIO()
    Image.fromarray(array.astype("uint8"), mode="RGB").save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode()


bands = stack.encode()
before, after = rgb, rgb.copy()
after[30:80, 30:80] = 255

checks = {
    "vqa":       ("/v1/vqa", {"question": "Is there water in this scene?", "bands": bands}),
    "caption":   ("/v1/vqa", {"task": "caption", "bands": bands}),
    "classify":  ("/v1/classify", {"bands": bands}),
    "grounding": ("/v1/grounding", {"query": "Highlight the water body", "imageB64": png_b64(rgb)}),
    "change":    ("/v1/change", {"question": "What changed?", "beforeB64": png_b64(before), "afterB64": png_b64(after)}),
    "fusion":    ("/v1/fusion", {"question": "What does SAR add over optical?", "bands": bands}),
}

for name, (path, payload) in checks.items():
    body = post(path, payload)
    if "error" in body:
        print(f"FAIL {name}: {body['error']} {body['detail']}\n")
        continue
    answer = body.get("answer") or f"{body.get('present')}"
    print(f"ok   {name:10s} [{body.get('elapsedMs', 0)} ms] {str(answer)[:150]}")
    if body.get("notes"):
        for note in body["notes"]:
            print(f"       note: {note[:150]}")
    print()

In [ ]:
# 10. Open the public tunnel. Kaggle cannot expose a port directly, so cloudflared
#     forwards one. The printed URL is what the backend .env needs.
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

TUNNEL_LOG = Path("/kaggle/working/tunnel.log")
tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=TUNNEL_LOG.open("w"),
    stderr=subprocess.STDOUT,
)

import re

public_url = None
for attempt in range(40):
    time.sleep(3)
    match = re.search(r"https://[-\w]+\.trycloudflare\.com", TUNNEL_LOG.read_text())
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print(TUNNEL_LOG.read_text()[-2000:])
    raise SystemExit("tunnel did not come up")

print(f"public URL: {public_url}\n")
print("Put this in backend/.env — all four point at the same server:\n")
for name in ("VLM", "GROUNDING", "CHANGE", "FUSION"):
    print(f"SATQUERY_{name}_ENDPOINT={public_url}")

In [ ]:
# 11. Verify the tunnel end to end, then keep the session alive.
with urllib.request.urlopen(f"{public_url}/health", timeout=30) as reply:
    remote = json.loads(reply.read())
print("tunnel reachable. loaded models:", sum(1 for m in remote["models"] if m["loaded"]))

print("\nKeeping the session alive. Interrupt the kernel to stop.")
print("Kaggle idles out after ~20 minutes of silence, so this loop keeps printing.\n")

try:
    minute = 0
    while True:
        time.sleep(60)
        minute += 1
        if server.poll() is not None:
            print("server died; last log lines:")
            print("\n".join(LOG.read_text().splitlines()[-20:]))
            break
        if minute % 5 == 0:
            print(f"alive {minute} min — {public_url}", flush=True)
except KeyboardInterrupt:
    print("stopping")
    tunnel.terminate()
    server.terminate()